In [3]:
from transformers import pipeline

# Load the 0.5B instruct model (lightweight & chat-capable)
chat_pipe = pipeline("text-generation",
                     model="Qwen/Qwen2.5-0.5B-Instruct",
                     device_map="auto")

def chat_with_bot(user_input):
    # System prompt sets the "personality" of the bot
    messages = [
        {"role": "system", "content": "You are a kind, empathetic AI assistant. If a user is having a bad day, offer gentle support and respect their space."},
        {"role": "user", "content": user_input},
    ]

    # Generate response
    output = chat_pipe(messages, max_new_tokens=100, do_sample=True, temperature=0.7)
    return output[0]['generated_text'][-1]['content']

# Test it
print(chat_with_bot("i had a very bad day please leave me alone"))


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


I'm sorry to hear that you're going through a difficult time. It's important to allow yourself some space to process what has happened and to prioritize your own well-being. Please know that I am here for you, no matter how big or small the issue may seem. You deserve love, care, and support, and I am committed to helping you through this. Let me know if there's anything specific you need assistance with or if you'd like to talk about something in particular.


In [8]:
from transformers import pipeline

# We must use "text-generation" to get ChatGPT-like behavior
# This uses the Qwen 0.5B model you downloaded earlier
chat_model = pipeline("text-generation",
                      model="Qwen/Qwen2.5-0.5B-Instruct",
                      device_map="auto")

def chatbot_chat(user_text):
    # This structure is required for "Instruct" models to talk correctly
    messages = [
        {"role": "system", "content": "You are a helpful and kind AI assistant."},
        {"role": "user", "content": user_text},
    ]

    # Generate the response
    output = chat_model(messages, max_new_tokens=128, temperature=0.7)

    # Extract the response text
    return output[0]['generated_text'][-1]['content']

# 2. Test it
user_input = input("Say something to the bot: ")
print(chatbot_chat(user_input))

Device set to use cpu


Say something to the bot: hey i had a very bad day im going to die
I'm sorry to hear that you're experiencing such a difficult situation. It's important to take care of yourself during this time, but it's also important to remember that there is hope for recovery.

If you feel like you need to talk about your feelings or if you would like support in the form of counseling or therapy, I can certainly provide that. However, it's important to prioritize self-care and take care of yourself before seeking professional help.

It's also worth noting that suicide is not a solution and should be avoided at all costs. If you have thoughts of harming yourself, please seek help immediately by calling 911 or


In [11]:
# ==============================================================
# 📦 Install required packages
#   (torch 2.3.0 + transformers 4.44.0 + accelerate 0.34.2)
#   Uncomment the bitsandbytes line if you need 8‑bit quantisation.
# ==============================================================

!pip install --quiet torch==2.3.0 \
               transformers==4.44.0 \
               accelerate==0.34.2 \
               sentencepiece==0.2.0
# !pip install --quiet bitsandbytes==0.44.1   # <-- uncomment for 8‑bit mode

# ==============================================================
# 🛠 Imports & optional HF token
# ==============================================================

import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# If you are using a *private* repo, set your token here:
# HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
HF_TOKEN = os.getenv("HF_TOKEN")  # reads from environment variable if you set one

# ==============================================================
# 🎛 Choose precision (fp16 = fast, needs ≥24 GB GPU)
#   For 12‑16 GB GPUs comment out the fp16 block and uncomment the 8‑bit block.
# ==============================================================

# ----- Full‑precision fp16 (recommended on big GPUs) -----
dtype = torch.float16
quant_cfg = None   # keep None for normal fp16

# ----- 8‑bit mode (good for 12‑16 GB GPUs) -------------
# dtype = torch.float16
# quant_cfg = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)

# ==============================================================
# 📥 Load tokenizer & model
# ==============================================================

model_name = "zai-org/GLM-4.7"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_auth_token=HF_TOKEN,
    trust_remote_code=True,          # required for GLM‑4.7's custom tokenizer
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",               # auto‑places layers on GPU/CPU
    torch_dtype=dtype,
    quantization_config=quant_cfg,
    use_auth_token=HF_TOKEN,
    trust_remote_code=True,
)

model.eval()                         # disable dropout etc.

# ==============================================================
# 💬 Build a chat prompt (feel free to edit the messages)
# ==============================================================

messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user",   "content": "Who are you?"},
    {"role": "user",   "content": "What is the capital of France?"},
]

# The template adds the special token that signals “assistant’s turn”
chat_inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,     # <-- very important!
    tokenize=True,
    return_tensors="pt",
    return_dict=True,
)

# Move tensors to the same device the model lives on (GPU if available)
chat_inputs = {k: v.to(model.device) for k, v in chat_inputs.items()}

# ==============================================================
# 🚀 Generate a response
# ==============================================================

output_ids = model.generate(
    **chat_inputs,
    max_new_tokens=120,              # how many tokens to generate *after* the prompt
    do_sample=True,                 # set False for greedy, deterministic output
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.02,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

# Strip away the prompt part and decode only the newly generated tokens
generated_ids = output_ids[0][chat_inputs["input_ids"].shape[-1]:]
answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("\n🤖 Assistant:", answer)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

ModuleNotFoundError: No module named 'transformers.models.glm4_moe'